# Projet final : descriptions de restaurants

GEM/E2E -> Phi-3 QLoRA -> comparaison base/adaptateurs -> ROUGE -> Vertex + Chainlit.

Ce notebook inspecte les vrais fichiers locaux ; il ne fabrique aucun resultat. Les cellules de lancement GPU sont laissees sous forme de commandes documentees.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from agentic_workflows.restaurant_data import restaurant_prompt
DATA = ROOT / 'data/restaurant'
print(ROOT)

## 1. Snapshot et splits officiels

Executer `scripts/prepare_restaurant_dataset.py` au prealable. Les revisions et empreintes rendent la source reproductible.

In [ ]:
manifest = json.loads((DATA / 'raw/manifest.json').read_text(encoding='utf-8'))
audit = json.loads((DATA / 'prepared/preparation.json').read_text(encoding='utf-8'))
print('Source:', manifest['source_revision'])
print('Parquet:', manifest['parquet_revision'])
print('Overlap:', audit['canonical_overlap_before_filtering'])
pd.DataFrame(audit['splits']).T

## 2. Passage attributs -> prompt/reponse

Le prompt ne contient pas la description cible. Les references sont utilisees pour la supervision ou pour la metrique, jamais comme entree du modele a evaluer.

In [ ]:
example = json.loads((ROOT / 'examples/restaurants.json').read_text(encoding='utf-8'))[0]
print(restaurant_prompt(example['attributes']))
train = pd.read_csv(DATA / 'prepared/train.csv')
print(json.dumps(json.loads(train.iloc[0]['messages']), ensure_ascii=False, indent=2))
validation = pd.read_csv(DATA / 'prepared/validation.csv')
validation[['meaning_representation', 'references']].head()

## 3. Compilation locale du graphe

Cela ne soumet aucun job Vertex et ne reserve aucun GPU.

In [ ]:
import subprocess
subprocess.run([sys.executable, 'scripts/run_restaurant_pipeline.py', '--compile-only'], cwd=ROOT, check=True)

## 4. Experiences sur Vertex

Depuis le terminal, commencer par `scripts/run_restaurant_pipeline.py --smoke-test`. Ce test ne mesure pas la qualite. Le pilote declare utilise 4 000 lignes train, 200 MR validation et une epoque. La validation officielle suit la loss et sert au choix du checkpoint.

Pour le bilan final, utiliser `--evaluate-model gs://... --test-limit 0` apres selection du modele. Ce second pipeline ne reentraine pas. Voir `docs/restaurant_protocol.md`.

## 5. Comparaison des sorties reelles

La base et les adaptateurs utilisent les memes entrees et parametres. Les scores utilisent la meilleure reference par metrique, puis une moyenne par MR.

In [ ]:
from agentic_workflows.restaurant_evaluation import score_restaurant_predictions
results_path = DATA / 'evaluation_results.csv'
if results_path.exists():
    results = pd.read_csv(results_path, keep_default_na=False)
    _, metrics = score_restaurant_predictions(results)
    print('N:', len(results), 'split:', results['source_split'].unique())
    display(pd.DataFrame([{
        'metric': kind, 'base': metrics.get('base_' + kind),
        'fine_tuned': metrics['finetuned_' + kind], 'delta': metrics.get('delta_' + kind)
    } for kind in ('rouge1', 'rouge2', 'rougeL')]))
    display(results[['meaning_representation', 'base_response', 'extracted_response']].head())
else:
    print('Aucun resultat de qualite telecharge. Ne pas utiliser le smoke test comme bilan final.')

## 6. Analyse qualitative

Examiner fluidite, attributs omis, faits ajoutes et negations. Tester un profil incomplet et `familyFriendly[no]`. ROUGE et la presence litterale des noms ne prouvent pas toute la fidelite semantique. Le pretraitement normalise l ordre des attributs, ce qui rend les permutations identiques pour le modele.

In [ ]:
examples = json.loads((ROOT / 'examples/restaurants.json').read_text(encoding='utf-8'))
assert restaurant_prompt(examples[0]['attributes']) == restaurant_prompt(examples[-1]['attributes'])
pd.DataFrame(examples)

## 7. Registre, application et presentation

Apres un vrai entrainement reussi : telecharger les artefacts, enregistrer le modele avec `register_model_with_custom_handler.py`, puis deployer avec `manage_endpoint.py deploy`. Tester l API avant de lancer `run_chat.py`. Les ressources Restaurant sont separees des anciennes ressources Yoda.

Demonstration : `docs/final_project_demo.md`. Pour terminer, arreter Chainlit puis executer `manage_endpoint.py cleanup`. Le quota T4 de serving est distinct de celui de l entrainement.